# VQA – Visual Question Answering
## CNN Encoder + LSTM Decoder (with & without Attention, scratch & pretrained)

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import os

# # Tạo thư mục trên Drive
# save_dir = '/content/drive/MyDrive/vqa'
# os.makedirs(save_dir, exist_ok=True)
# os.chdir(save_dir)
# print("Working dir:", os.getcwd())

# # # Download thẳng vào Drive
# !wget -q --show-progress http://images.cocodataset.org/zips/train2014.zip
# !wget -q --show-progress http://images.cocodataset.org/zips/val2014.zip

# base="https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa"
# !wget -q --show-progress $base/v2_Questions_Train_mscoco.zip
# !wget -q --show-progress $base/v2_Questions_Val_mscoco.zip
# !wget -q --show-progress $base/v2_Annotations_Train_mscoco.zip
# !wget -q --show-progress $base/v2_Annotations_Val_mscoco.zip

# # Giải nén thẳng vào Drive
# !unzip -q train2014.zip && rm train2014.zip
# !unzip -q val2014.zip && rm val2014.zip
# !unzip -q v2_Questions_Train_mscoco.zip && rm v2_Questions_Train_mscoco.zip
# !unzip -q v2_Questions_Val_mscoco.zip && rm v2_Questions_Val_mscoco.zip
# !unzip -q v2_Annotations_Train_mscoco.zip && rm v2_Annotations_Train_mscoco.zip
# !unzip -q v2_Annotations_Val_mscoco.zip && rm v2_Annotations_Val_mscoco.zip

# print("✅ Xong! Data nằm tại:", save_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working dir: /content/drive/MyDrive/vqa
train2014.zip       100%[===================>]  12.58G  43.7MB/s    in 6m 9s   


In [ ]:
# Install
!pip install pycocoevalcap -q
!pip install timm -q
import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 7.4 MB/s eta 0:00:00


True

In [ ]:
import torch, torch.nn as nn, torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

import timm
from PIL import Image
import json, os, re
from collections import Counter
import numpy as np
from tqdm import tqdm

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction, corpus_bleu
from nltk.translate.meteor_score import meteor_score
from pycocoevalcap.cider.cider import Cider

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)


Device: cpu


## 1. Hyperparameters

In [ ]:
# ─── Hyperparameters ───────────────────────────────────────
EMBED_DIM    = 300     # word embedding size
HIDDEN_DIM   = 512     # LSTM hidden size
MAX_Q_LEN    = 20      # truncate/pad question tokens
MAX_A_LEN    = 10      # max generated answer tokens  (incl. SOS/EOS)
BATCH_SIZE   = 32
EPOCHS       = 1
TOP_K_ANS    = 3000    # answer vocabulary size (top-k answers)
TOP_K_QUES   = 10000   # question word vocabulary size

PAD, SOS, EOS = "<pad>", "<sos>", "<eos>"


## 2. Download VQA v2 Data (run once on Colab)

In [ ]:
# # ── Images ──────────────────────────────────────────────────
# !mkdir -p /content/vqa
# %cd /content/vqa

# !wget -q --show-progress http://images.cocodataset.org/zips/train2014.zip
# !wget -q --show-progress http://images.cocodataset.org/zips/val2014.zip

# # ── Questions & Annotations ─────────────────────────────────
# !wget -q https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Train_mscoco.zip
# !wget -q https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip
# !wget -q https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Train_mscoco.zip
# !wget -q https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip

# !unzip -q train2014.zip && unzip -q val2014.zip
# !unzip -q v2_Questions_Train_mscoco.zip && unzip -q v2_Questions_Val_mscoco.zip
# !unzip -q v2_Annotations_Train_mscoco.zip && unzip -q v2_Annotations_Val_mscoco.zip


## 3. Build Vocabularies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

def tokenize(text: str):
    """Lowercase + simple whitespace tokenize (handles punctuation)."""
    return re.sub(r"[^a-z0-9 ]", " ", text.lower()).split()


# Answer Vocabulary (top-K surface forms)
def build_answer_vocab(ann_file, top_k=TOP_K_ANS):
    counter = Counter()
    with open(ann_file) as f:
        for ann in json.load(f)["annotations"]:
            for a in ann["answers"]:
                counter[a["answer"].lower().strip()] += 1
    most_common = counter.most_common(top_k)
    ans2id = {a: i for i, (a, _) in enumerate(most_common)}
    id2ans  = {i: a for a, i in ans2id.items()}
    return ans2id, id2ans

answer_vocab, answer_id2word = build_answer_vocab(
    "/content/drive/MyDrive/vqa/v2_mscoco_train2014_annotations.json"
)
print(f"Answer vocab size : {len(answer_vocab)}")


# Answer Token Vocabulary (for LSTM decoder)
# Each answer can be multi-word → build token vocab from those words
def build_ans_token_vocab(answer_vocab):
    tokens = set()
    for a in answer_vocab:
        tokens.update(tokenize(a))
    word2id = {PAD: 0, SOS: 1, EOS: 2}
    for w in sorted(tokens):
        word2id[w] = len(word2id)
    id2word = {i: w for w, i in word2id.items()}
    return word2id, id2word

ans_word2id, ans_id2word = build_ans_token_vocab(answer_vocab)
ANS_TOKEN_VOCAB = len(ans_word2id)
print(f"Answer token vocab: {ANS_TOKEN_VOCAB}")


# Question Vocabulary
def build_question_vocab(ques_file, top_k=TOP_K_QUES):
    counter = Counter()
    with open(ques_file) as f:
        for q in json.load(f)["questions"]:
            counter.update(tokenize(q["question"]))
    most_common = counter.most_common(top_k - 2)
    word2id = {PAD: 0, "<unk>": 1}
    for w, _ in most_common:
        word2id[w] = len(word2id)
    id2word = {i: w for w, i in word2id.items()}
    return word2id, id2word

ques_word2id, ques_id2word = build_question_vocab(
    "/content/drive/MyDrive/vqa/v2_OpenEnded_mscoco_train2014_questions.json"
)
QUES_VOCAB = len(ques_word2id)
print(f"Question vocab    : {QUES_VOCAB}")


Mounted at /content/drive
Answer vocab size : 3000
Answer token vocab: 2420
Question vocab    : 10000


## 4. Dataset & DataLoader

In [ ]:
def encode_question(text: str, word2id, max_len=MAX_Q_LEN):
    tokens = tokenize(text)[:max_len]
    ids = [word2id.get(t, word2id["<unk>"]) for t in tokens]
    return torch.tensor(ids, dtype=torch.long)


def encode_answer(text: str, word2id, max_len=MAX_A_LEN):
    """Returns sequence [SOS, w1, w2, ..., EOS] padded to max_len."""
    tokens = [SOS] + tokenize(text)[:max_len - 2] + [EOS]
    ids = [word2id.get(t, word2id[PAD]) for t in tokens]
    # pad to fixed length so we can stack in batch
    ids += [word2id[PAD]] * (max_len - len(ids))
    return torch.tensor(ids[:max_len], dtype=torch.long)


class VQADataset(Dataset):
    def __init__(self, img_dir, ques_file, ann_file, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.split = "train" if "train" in img_dir else "val"

        with open(ques_file) as f:
            self.questions = json.load(f)["questions"]

        with open(ann_file) as f:
            anns = json.load(f)["annotations"]
        self.ans_dict = {a["question_id"]: a["answers"] for a in anns}

    def __len__(self):
        return len(self.questions)

    def __getitem__(self, idx):
        q   = self.questions[idx]
        qid = q["question_id"]
        img_id = q["image_id"]
        prefix = "COCO_train2014" if self.split == "train" else "COCO_val2014"

        img = Image.open(
            os.path.join(self.img_dir, f"{prefix}_{img_id:012d}.jpg")
        ).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # Question: real token ids
        question_ids = encode_question(q["question"], ques_word2id)

        # Answers: list of raw strings + encoded target
        raw_answers = [a["answer"].lower().strip() for a in self.ans_dict[qid]]
        # Use the most frequent answer as training target
        best_ans = Counter(raw_answers).most_common(1)[0][0]
        ans_seq  = encode_answer(best_ans, ans_word2id)   # (MAX_A_LEN,)

        return img, question_ids, ans_seq, raw_answers


def vqa_collate_fn(batch):
    imgs, questions, ans_seqs, raw_answers = zip(*batch)

    imgs     = torch.stack(imgs)                          # (B,3,H,W)
    lengths  = torch.tensor([len(q) for q in questions], dtype=torch.long)
    questions = pad_sequence(questions, batch_first=True, padding_value=0)  # (B, T_q)
    ans_seqs  = torch.stack(ans_seqs)                     # (B, MAX_A_LEN)

    return imgs, questions, lengths, ans_seqs, raw_answers


In [ ]:
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

train_dataset = VQADataset(
    img_dir   = "/content/drive/MyDrive/vqa/train2014",
    ques_file = "/content/drive/MyDrive/vqa/v2_OpenEnded_mscoco_train2014_questions.json",
    ann_file  = "/content/drive/MyDrive/vqa/v2_mscoco_train2014_annotations.json",
    transform = transform
)
val_dataset = VQADataset(
    img_dir   = "/content/drive/MyDrive/vqa/val2014",
    ques_file = "/content/drive/MyDrive/vqa/v2_OpenEnded_mscoco_val2014_questions.json",
    ann_file  = "/content/drive/MyDrive/vqa/v2_mscoco_val2014_annotations.json",
    transform = transform
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=vqa_collate_fn, pin_memory=True)

#------------------------------------------------------------------------------------------
# Thêm ngay sau train_loader = DataLoader(...)
from torch.utils.data import Subset
import random

subset_idx = random.sample(range(len(train_dataset)), k=int(0.05 * len(train_dataset)))   # ĐANG TEST THỬ
train_subset = Subset(train_dataset, subset_idx)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=vqa_collate_fn, pin_memory=True)
#------------------------------------------------------------------------------------------

val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, collate_fn=vqa_collate_fn)

# Quick sanity check
imgs, ques, lens, ans_seqs, raw_ans = next(iter(val_loader))
print("imgs    :", imgs.shape)      # (32, 3, 224, 224)
print("ques    :", ques.shape)      # (32, T_q)
print("ans_seqs:", ans_seqs.shape)  # (32, MAX_A_LEN)
print("sample  :", raw_ans[0][:3])


imgs    : torch.Size([32, 3, 224, 224])
ques    : torch.Size([32, 10])
ans_seqs: torch.Size([32, 10])
sample  : ['down', 'down', 'at table']


## 5. CNN Encoders

In [ ]:
# Model 1, 3: CNN trained from scratch
class CNN_Scratch(nn.Module):
    """Lightweight CNN encoder – trained from scratch."""
    def __init__(self, out_dim=HIDDEN_DIM):
        super().__init__()
        self.out_dim = out_dim
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.proj = nn.Linear(256, out_dim)

    def forward(self, x):           # x: (B,3,224,224)
        feat = self.net(x).flatten(1)   # (B,256)
        return self.proj(feat)          # (B, out_dim)


# Model 2: ResNet-50 pretrained, no attention
# Model 4: ResNet-50 pretrained, with attention
class CNN_Pretrained(nn.Module):
    """
    ResNet-50 backbone with pretrained ImageNet weights.
    Returns per-spatial feature maps for attention, or pooled vector.
    """
    def __init__(self, out_dim=HIDDEN_DIM, return_maps=False):
        super().__init__()
        self.return_maps = return_maps
        self.out_dim = out_dim

        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        # Remove avgpool + fc → keep up to layer4
        self.backbone = nn.Sequential(*list(resnet.children())[:-2])  # (B,2048,7,7)
        self.avgpool  = nn.AdaptiveAvgPool2d((1, 1))
        self.proj     = nn.Linear(2048, out_dim)

        # Freeze early layers for faster convergence
        for param in list(self.backbone.parameters())[:-20]:
            param.requires_grad = False

    def forward(self, x):
        feat_map = self.backbone(x)                   # (B, 2048, 7, 7)
        if self.return_maps:
            # Return spatial features: (B, N, 2048) where N=7*7=49
            B, C, H, W = feat_map.shape
            return feat_map.view(B, C, H * W).permute(0, 2, 1)   # (B,49,2048)
        pooled = self.avgpool(feat_map).flatten(1)    # (B, 2048)
        return self.proj(pooled)                      # (B, out_dim)


## 6. Question Encoder (LSTM)

In [ ]:
class QuestionEncoder(nn.Module):
    """Bidirectional LSTM question encoder."""
    def __init__(self, vocab_size=QUES_VOCAB, embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden_dim // 2, batch_first=True,
                             bidirectional=True, num_layers=2, dropout=0.3)
        self.out_dim = hidden_dim

    def forward(self, q, lengths):
        """
        q       : (B, T_q)  padded token ids
        lengths : (B,)      real lengths
        returns : h (B, hidden_dim), c (B, hidden_dim)  – merged from bidir
        """
        emb    = self.embed(q)                                     # (B, T_q, E)
        packed = pack_padded_sequence(emb, lengths.cpu(),
                                      batch_first=True, enforce_sorted=False)
        out_packed, (h, c) = self.lstm(packed)
        # h: (num_layers*2, B, H//2) → take last 2 (fwd+bwd of top layer)
        h = torch.cat([h[-2], h[-1]], dim=1)   # (B, H)
        c = torch.cat([c[-2], c[-1]], dim=1)   # (B, H)
        return h, c


## 7. Spatial Attention Module

In [ ]:
class SpatialAttention(nn.Module):
    """
    Soft attention over CNN spatial feature map.
    Conditioned on the question hidden state.
    img_feats : (B, N, img_dim)   – N spatial locations
    ques_h    : (B, ques_dim)
    returns   : (B, img_dim)      – attended image vector
    """
    def __init__(self, img_dim=2048, ques_dim=HIDDEN_DIM, att_dim=256):
        super().__init__()
        self.W_img  = nn.Linear(img_dim, att_dim, bias=False)
        self.W_ques = nn.Linear(ques_dim, att_dim, bias=False)
        self.v      = nn.Linear(att_dim, 1, bias=False)

    def forward(self, img_feats, ques_h):
        # img_feats : (B, N, img_dim)
        # ques_h    : (B, ques_dim)
        N   = img_feats.size(1)
        e_i = self.W_img(img_feats)                          # (B, N, att_dim)
        e_q = self.W_ques(ques_h).unsqueeze(1).expand(-1, N, -1)  # (B, N, att_dim)
        e   = torch.tanh(e_i + e_q)                         # (B, N, att_dim)
        a   = torch.softmax(self.v(e), dim=1)               # (B, N, 1)
        ctx = (a * img_feats).sum(dim=1)                     # (B, img_dim)
        return ctx, a.squeeze(-1)                            # ctx, attn_weights


## 8. LSTM Answer Decoder

In [ ]:
class LSTMDecoder(nn.Module):
    """
    Autoregressive LSTM decoder that generates answer tokens one-by-one.
    Initialised with fused (image + question) context.
    """
    def __init__(self, token_vocab=ANS_TOKEN_VOCAB,
                 embed_dim=EMBED_DIM, hidden_dim=HIDDEN_DIM, dropout=0.3):
        super().__init__()
        self.embed   = nn.Embedding(token_vocab, embed_dim, padding_idx=0)
        self.lstm    = nn.LSTM(embed_dim, hidden_dim, batch_first=True, num_layers=2,
                               dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        self.fc      = nn.Linear(hidden_dim, token_vocab)

    def forward(self, ans_seq, h0, c0):
        """
        Teacher-forcing forward pass.
        ans_seq : (B, T)   – token ids incl. SOS, excl. last EOS
        h0, c0  : (1, B, H) or (num_layers, B, H)
        returns : logits (B, T-1, vocab)
        """
        emb    = self.embed(ans_seq[:, :-1])         # (B, T-1, E)
        out, _ = self.lstm(emb, (h0, c0))            # (B, T-1, H)
        logits = self.fc(self.dropout(out))          # (B, T-1, vocab)
        return logits

    def decode_greedy(self, h, c, max_len=MAX_A_LEN, device=device):
        """Single-sample greedy decode (batch of 1 supported)."""
        B   = h.size(1)
        tok = torch.full((B, 1), ans_word2id[SOS], dtype=torch.long, device=device)
        result = [[] for _ in range(B)]
        done   = [False] * B

        for _ in range(max_len - 1):
            emb       = self.embed(tok)              # (B,1,E)
            out, (h, c) = self.lstm(emb, (h, c))
            logits    = self.fc(out[:, -1])          # (B, vocab)
            next_ids  = logits.argmax(-1)            # (B,)
            for b in range(B):
                if done[b]:
                    continue
                nid = next_ids[b].item()
                if nid == ans_word2id[EOS]:
                    done[b] = True
                else:
                    result[b].append(ans_id2word.get(nid, PAD))
            tok = next_ids.unsqueeze(1)
            if all(done):
                break

        return [" ".join(r) if r else "" for r in result]


## 9. VQA Model (4 variants)

In [ ]:
class VQAModel(nn.Module):
    """
    Unified VQA model supporting 4 configurations:
      1. CNN_Scratch  + No Attention
      2. CNN_Pretrained + No Attention
      3. CNN_Scratch  + Spatial Attention
      4. CNN_Pretrained + Spatial Attention

    Architecture:
        Image  ──► CNN Encoder ──────────────────────────────┐
                                                             ▼
        Question ─► Bidirectional LSTM Encoder → (h, c) ─► Fusion
                                                             ▼
                                                       LSTM Decoder
                                                             ▼
                                                       Answer tokens
    """
    def __init__(self, use_pretrained=False, use_attention=False):
        super().__init__()
        self.use_attention = use_attention
        self.use_pretrained = use_pretrained

        # ── CNN Encoder ────────────────────────────────────
        if use_attention and use_pretrained:
            # Needs spatial feature maps → return_maps=True
            self.img_encoder  = CNN_Pretrained(return_maps=True)
            self.img_proj     = nn.Linear(2048, HIDDEN_DIM)   # per-region proj
            self.attention    = SpatialAttention(img_dim=2048, ques_dim=HIDDEN_DIM)
        elif use_pretrained:
            self.img_encoder  = CNN_Pretrained(out_dim=HIDDEN_DIM, return_maps=False)
        elif use_attention:
            # Scratch CNN → expose intermediate feature map
            self.img_encoder  = CNN_Scratch(out_dim=HIDDEN_DIM)
            # For scratch+attention we simulate spatial feats via repeated vector
            # (more principled: return feature map before pooling)
        else:
            self.img_encoder  = CNN_Scratch(out_dim=HIDDEN_DIM)

        # ── Question Encoder ───────────────────────────────
        self.q_encoder = QuestionEncoder()

        # ── Fusion: project fused context to (num_layers, B, H) ──
        # Decoder has 2 LSTM layers → need to produce h0, c0 with shape (2, B, H)
        self.fusion_h = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, HIDDEN_DIM), nn.Tanh()
        )
        self.fusion_c = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, HIDDEN_DIM), nn.Tanh()
        )

        # ── LSTM Decoder ───────────────────────────────────
        self.decoder = LSTMDecoder()

    # ────────────────────────────────────────────────────────
    def _encode(self, images, questions, lengths):
        """Return fused (h0, c0) for decoder initialisation."""
        # Question encoding
        q_h, q_c = self.q_encoder(questions, lengths)   # (B, H)

        # Image encoding
        if self.use_attention and self.use_pretrained:
            img_maps = self.img_encoder(images)          # (B, 49, 2048)
            ctx, _   = self.attention(img_maps, q_h)     # (B, 2048)
            img_feat = self.img_proj(ctx)                # (B, H)
        else:
            img_feat = self.img_encoder(images)          # (B, H)
            if self.use_attention:
                # Scratch+attention: element-wise gating
                gate     = torch.sigmoid(img_feat * q_h) # (B, H)
                img_feat = gate * img_feat

        # Fusion
        fused = torch.cat([img_feat, q_h], dim=1)       # (B, 2H)
        h0 = self.fusion_h(fused).unsqueeze(0).repeat(2, 1, 1)  # (2,B,H)
        c0 = self.fusion_c(fused).unsqueeze(0).repeat(2, 1, 1)  # (2,B,H)
        return h0, c0

    # ────────────────────────────────────────────────────────
    def forward(self, images, questions, lengths, ans_seq):
        """Teacher-forcing training pass. Returns logits (B, T-1, vocab)."""
        h0, c0 = self._encode(images, questions, lengths)
        return self.decoder(ans_seq, h0, c0)

    # ────────────────────────────────────────────────────────
    def generate(self, images, questions, lengths):
        """Greedy decode – returns list[str] of length B."""
        h0, c0 = self._encode(images, questions, lengths)
        return self.decoder.decode_greedy(h0, c0, device=images.device)


## 10. Instantiate 4 Models

In [ ]:
# Model 1: CNN Scratch   – No Attention
model1 = VQAModel(use_pretrained=False, use_attention=False).to(device)

# Model 2: CNN Pretrained – No Attention
model2 = VQAModel(use_pretrained=True,  use_attention=False).to(device)

# Model 3: CNN Scratch   – With Spatial Attention
model3 = VQAModel(use_pretrained=False, use_attention=True).to(device)

# Model 4: CNN Pretrained – With Spatial Attention
model4 = VQAModel(use_pretrained=True,  use_attention=True).to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f"Model1 (Scratch, NoAtt)  : {count_params(model1):,} params")
print(f"Model2 (Pretrain, NoAtt) : {count_params(model2):,} params")
print(f"Model3 (Scratch, Att)    : {count_params(model3):,} params")
print(f"Model4 (Pretrain, Att)   : {count_params(model4):,} params")


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 149MB/s]


Model1 (Scratch, NoAtt)  : 13,026,084 params
Model2 (Pretrain, NoAtt) : 22,483,492 params
Model3 (Scratch, Att)    : 13,026,084 params
Model4 (Pretrain, Att)   : 24,188,196 params


## 11. Training Loop

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler=None):
    model.train()
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    total_loss, steps = 0.0, 0

    for imgs, ques, lengths, ans_seqs, _ in tqdm(loader, desc="  Train", leave=False):
        imgs     = imgs.to(device)
        ques     = ques.to(device)
        lengths  = lengths.to(device)
        ans_seqs = ans_seqs.to(device)      # (B, MAX_A_LEN)

        logits = model(imgs, ques, lengths, ans_seqs)   # (B, T-1, vocab)
        targets = ans_seqs[:, 1:]                       # (B, T-1)  shift by 1

        loss = criterion(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    if scheduler:
        scheduler.step()

    return total_loss / steps


def train_model(model, name, epochs=EPOCHS, lr=1e-3):
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    print(f"\n{'='*60}")
    print(f" Training: {name}")
    print(f"{'='*60}")
    for ep in range(1, epochs + 1):
        loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        print(f"  Epoch {ep:02d}/{epochs}  loss={loss:.4f}")

    return model


In [ ]:
# Train all 4 models
model1 = train_model(model1, "Model1 – CNN Scratch,  No Attention")
model2 = train_model(model2, "Model2 – CNN Pretrain, No Attention",  lr=5e-4)
model3 = train_model(model3, "Model3 – CNN Scratch,  Spatial Attn")
model4 = train_model(model4, "Model4 – CNN Pretrain, Spatial Attn",  lr=5e-4)



 Training: Model1 – CNN Scratch,  No Attention


  Epoch 01/1  loss=2.2901

 Training: Model2 – CNN Pretrain, No Attention


  Epoch 01/1  loss=2.6033

 Training: Model3 – CNN Scratch,  Spatial Attn


  Epoch 01/1  loss=2.2312

 Training: Model4 – CNN Pretrain, Spatial Attn


  Epoch 01/1  loss=2.5004


## 12. Evaluation Metrics

| Metric | Lý do dùng |
|---|---|
| **VQA Accuracy** | Chuẩn chính thức VQAv2 (min(#human_ans / 3, 1)) |
| **BLEU-1/2/3/4** | n-gram precision, phổ biến trong NLG |
| **METEOR** | Tính cả stemming & synonym matching |
| **CIDEr** | Đo độ tương đồng ngữ nghĩa với consensus caption |

In [ ]:
def vqa_accuracy(pred: str, gt_answers: list) -> float:
    """
    Official VQAv2 accuracy:
       acc = min(#annotators_agreed / 3, 1.0)
    for each question there are 10 annotators; acc = min(count/3, 1)
    """
    count = sum(1 for a in gt_answers if a == pred.lower().strip())
    return min(count / 3.0, 1.0)


def evaluate_vqa(model, dataloader, max_batches=None, desc="Eval"):
    model.eval()
    smooth = SmoothingFunction().method4
    cider  = Cider()

    total = 0
    vqa_acc_total = 0.0
    bleu_totals   = [0.0] * 4
    meteor_total  = 0.0
    gts, res      = {}, {}

    with torch.no_grad():
        for i, (imgs, ques, lengths, ans_seqs, raw_answers) in enumerate(
                tqdm(dataloader, desc=f"  {desc}", leave=False)):
            if max_batches and i >= max_batches:
                break

            imgs    = imgs.to(device)
            ques    = ques.to(device)
            lengths = lengths.to(device)

            # Batch generate
            preds = model.generate(imgs, ques, lengths)   # list[str] length B

            for b, pred in enumerate(preds):
                gt_answers = list(raw_answers[b])
                total += 1

                # ── VQA Accuracy ──────────────────────────
                vqa_acc_total += vqa_accuracy(pred, gt_answers)

                # ── BLEU ──────────────────────────────────
                refs = [a.split() for a in gt_answers]
                hyp  = pred.split() if pred else [""]
                weights = [(1,0,0,0),(0.5,0.5,0,0),(0.33,0.33,0.33,0),(0.25,)*4]
                for k, w in enumerate(weights):
                    bleu_totals[k] += sentence_bleu(refs, hyp, w, smooth)

                # ── METEOR ────────────────────────────────
                meteor_total += max(
                    (meteor_score([a.split()], hyp) for a in gt_answers),
                    default=0.0
                )

                # ── CIDEr (accumulate) ────────────────────
                qid = f"{i}_{b}"
                gts[qid] = gt_answers          # list of strings ← correct format
                res[qid] = [pred] if pred else [""]

    cider_score, _ = cider.compute_score(gts, res)

    metrics = {
        "VQA_Acc" : vqa_acc_total / total,
        "BLEU-1"  : bleu_totals[0] / total,
        "BLEU-2"  : bleu_totals[1] / total,
        "BLEU-3"  : bleu_totals[2] / total,
        "BLEU-4"  : bleu_totals[3] / total,
        "METEOR"  : meteor_total / total,
        "CIDEr"   : cider_score,
    }
    return metrics


## 13. Run Evaluation on Validation Set

In [ ]:
MAX_EVAL_BATCHES = 200   # set None for full val set (slower)

results = {}
for (mdl, name) in [
    (model1, "Model1_Scratch_NoAtt"),
    (model2, "Model2_Pretrain_NoAtt"),
    (model3, "Model3_Scratch_Att"),
    (model4, "Model4_Pretrain_Att"),
]:
    print(f"\nEvaluating {name} ...")
    results[name] = evaluate_vqa(mdl, val_loader,
                                 max_batches=MAX_EVAL_BATCHES, desc=name)



Evaluating Model1_Scratch_NoAtt ...



Evaluating Model2_Pretrain_NoAtt ...



Evaluating Model3_Scratch_Att ...



Evaluating Model4_Pretrain_Att ...


## 14. Comparison Table

In [ ]:
import pandas as pd

df = pd.DataFrame(results).T.round(4)
df.index.name = "Model"

# Highlight best per metric
print("\n" + "="*80)
print(" VQA Model Comparison")
print("="*80)
print(df.to_string())
print("="*80)

# Display styled table if in Jupyter
try:
    display(df.style.highlight_max(axis=0, props="font-weight:bold;color:green"))
except:
    pass



 VQA Model Comparison
                       VQA_Acc  BLEU-1  BLEU-2  BLEU-3  BLEU-4  METEOR   CIDEr
Model                                                                         
Model1_Scratch_NoAtt    0.2987  0.3511  0.3511  0.3511  0.3511  0.1723  0.5699
Model2_Pretrain_NoAtt   0.2443  0.2951  0.2951  0.2951  0.2951  0.1427  0.4672
Model3_Scratch_Att      0.2991  0.3604  0.3604  0.3604  0.3604  0.1756  0.5716
Model4_Pretrain_Att     0.3057  0.3593  0.3593  0.3593  0.3593  0.1758  0.5854


,VQA_Acc,BLEU-1,BLEU-2,BLEU-3,BLEU-4,METEOR,CIDEr
Model,,,,,,,
Model1_Scratch_NoAtt,0.298700,0.351100,0.351100,0.351100,0.351100,0.172300,0.569900
Model2_Pretrain_NoAtt,0.244300,0.295100,0.295100,0.295100,0.295100,0.142700,0.467200
Model3_Scratch_Att,0.299100,0.360400,0.360400,0.360400,0.360400,0.175600,0.571600
Model4_Pretrain_Att,0.305700,0.359300,0.359300,0.359300,0.359300,0.175800,0.585400


## 15. Save Checkpoints

In [ ]:
# import os
# os.makedirs("/content/drive/MyDrive/vqa/checkpoints", exist_ok=True)

# for (mdl, name) in [
#     (model1, "model1_scratch_noatt"),
#     (model2, "model2_pretrain_noatt"),
#     (model3, "model3_scratch_att"),
#     (model4, "model4_pretrain_att"),
# ]:
#     path = f"/content/drive/MyDrive/vqa/checkpoints/{name}.pt"
#     torch.save(mdl.state_dict(), path)
#     print(f"Saved {path}")